# 第2回 発展課題：用語集のSFTと検索を比較する

同じ15問を、①学習前・資料なし、②LoRA学習後・資料なし、③学習前・用語集検索あり、で比較します。

1. Codespacesで `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` を実行し、116行のファイルをダウンロードする。
2. このノートブックをColabで開き、T4 GPUを設定してセッションを再起動する。
3. 上から順に実行。セクション2で、作成した glossary_sft.jsonl をアップロードする。
4. 最後に結果ZIPと、出力付きのこのipynbをダウンロードして提出する。

元のスクリプトが生成する正解は**自然文の定義**です。この版も自然文で回答させ、JSON形式の評価とは分けます。元の116件の質問・回答は変えず、学習時には共通のsystem指示だけを先頭に加えます。

モデルは同じQwen/Qwen3.5-0.8B、量子化なしのLoRAです。データ件数が増えるため3エポックにします（約45ステップ）。時間は環境により異なります。

用語集と検索関数は ykuma777/lecture-ai-engineering-2026 のlec02から取得したスナップショットです。ノートブックは未実行です。

## 0. GPUとライブラリ

T4 GPUを選び、2つのセルを順番に実行します。必要な版はこれまでと同じです。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定と用語集のスナップショット

学習率・LoRAランクはこれまでと同じで、学習回数は3エポック。評価は貪欲生成で固定します。
検索条件でも同じ基盤モデル・同じ生成設定を使います。ここではまだ学習しません。

In [3]:
from pathlib import Path
import json, time, hashlib, unicodedata

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42, "num_train_epochs": 3,
    "lr": 2e-4, "lora_r": 16, "lora_alpha": 32, "warmup_steps": 6,
    "max_seq_len": 512, "max_new_tokens": 160,
    "gen": {"do_sample": False},
}
WORK = Path("/content/glossary_sft_lab")
WORK.mkdir(parents=True, exist_ok=True)
SYSTEM = "あなたはこの講座の用語集の回答係です。用語の意味を日本語で簡潔に説明してください。用語集に記載がない、または確信が持てない場合は、推測せず『確認が必要です』と答えてください。"

GLOSSARY = {'version': '2026-draft-1', 'note': 'AIE2026 第1〜3回・第5回の用語集（たたき台）。講義スライド確定後に内容を差し替える。', 'terms': [{'id': 'ai-engineering', 'term': 'AIエンジニアリング', 'aliases': ['AI Engineering'], 'lecture': 1, 'category': '全体像', 'definition': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。', 'related': ['llm', 'evaluation-loop']}, {'id': 'evaluation-loop', 'term': '評価と改善のサイクル', 'aliases': ['改善サイクル'], 'lecture': 1, 'category': '全体像', 'definition': '作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。', 'related': ['ai-engineering', 'hallucination']}, {'id': 'llm', 'term': 'LLM（大規模言語モデル）', 'aliases': ['大規模言語モデル', 'Large Language Model'], 'lecture': 1, 'category': 'LLM基礎', 'definition': '大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。', 'related': ['token', 'prompt', 'pretraining']}, {'id': 'token', 'term': 'トークン', 'aliases': ['token'], 'lecture': 1, 'category': 'LLM基礎', 'definition': 'LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。', 'related': ['context-window', 'rate-limit']}, {'id': 'context-window', 'term': 'コンテキストウィンドウ', 'aliases': ['コンテキスト長', 'context window'], 'lecture': 1, 'category': 'LLM基礎', 'definition': 'LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。', 'related': ['token', 'agent-loop']}, {'id': 'prompt', 'term': 'プロンプト', 'aliases': ['prompt'], 'lecture': 1, 'category': 'LLM基礎', 'definition': 'LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。', 'related': ['system-prompt', 'llm']}, {'id': 'system-prompt', 'term': 'システムプロンプト', 'aliases': ['指示文', 'instructions', 'system instruction'], 'lecture': 1, 'category': 'LLM基礎', 'definition': '利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。', 'related': ['prompt', 'guardrail']}, {'id': 'temperature', 'term': 'temperature', 'aliases': ['温度パラメータ'], 'lecture': 1, 'category': 'LLM基礎', 'definition': '次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。', 'related': ['llm', 'evaluation-loop']}, {'id': 'hallucination', 'term': 'ハルシネーション', 'aliases': ['幻覚', 'hallucination'], 'lecture': 1, 'category': 'LLM基礎', 'definition': 'LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。', 'related': ['tool', 'evaluation-loop']}, {'id': 'api', 'term': 'API', 'aliases': ['Application Programming Interface', 'Web API'], 'lecture': 1, 'category': '開発基礎', 'definition': 'プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。', 'related': ['api-key', 'rate-limit']}, {'id': 'api-key', 'term': 'APIキー', 'aliases': ['API key'], 'lecture': 1, 'category': '開発基礎', 'definition': 'APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。', 'related': ['codespaces-secret', 'environment-variable']}, {'id': 'environment-variable', 'term': '環境変数', 'aliases': ['environment variable', 'env'], 'lecture': 1, 'category': '開発基礎', 'definition': 'プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。', 'related': ['api-key', 'codespaces-secret']}, {'id': 'rate-limit', 'term': 'レート制限', 'aliases': ['rate limit', '429エラー'], 'lecture': 1, 'category': '開発基礎', 'definition': '一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。', 'related': ['api', 'agent-loop', 'default-model']}, {'id': 'codespaces', 'term': 'GitHub Codespaces', 'aliases': ['Codespaces', 'コードスペース'], 'lecture': 1, 'category': '開発環境', 'definition': 'GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。', 'related': ['devcontainer', 'codespaces-secret']}, {'id': 'devcontainer', 'term': 'devcontainer', 'aliases': ['Dev Container', 'devcontainer.json'], 'lecture': 1, 'category': '開発環境', 'definition': '開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。', 'related': ['codespaces']}, {'id': 'codespaces-secret', 'term': 'Codespaces secret', 'aliases': ['Codespacesのシークレット'], 'lecture': 1, 'category': '開発環境', 'definition': 'GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。', 'related': ['api-key', 'environment-variable', 'codespaces']}, {'id': 'fork', 'term': 'fork', 'aliases': ['フォーク'], 'lecture': 1, 'category': 'Git/GitHub', 'definition': '他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。', 'related': ['branch', 'pull-request']}, {'id': 'branch', 'term': 'ブランチ', 'aliases': ['branch'], 'lecture': 1, 'category': 'Git/GitHub', 'definition': '作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。', 'related': ['pull-request', 'fork']}, {'id': 'pull-request', 'term': 'プルリクエスト（PR）', 'aliases': ['PR', 'Pull Request'], 'lecture': 1, 'category': 'Git/GitHub', 'definition': 'あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。', 'related': ['branch', 'answer-md']}, {'id': 'github-copilot', 'term': 'GitHub Copilot', 'aliases': ['Copilot'], 'lecture': 1, 'category': 'AIコーディング', 'definition': 'GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。', 'related': ['code-completion', 'coding-agent']}, {'id': 'code-completion', 'term': 'コード補完', 'aliases': ['インライン補完', 'inline suggestion'], 'lecture': 1, 'category': 'AIコーディング', 'definition': '書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。', 'related': ['github-copilot']}, {'id': 'coding-agent', 'term': 'コーディングエージェント', 'aliases': ['Coding Agent', 'AIコーディングエージェント'], 'lecture': 1, 'category': 'AIコーディング', 'definition': 'ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。', 'related': ['ai-agent', 'harness']}, {'id': 'pretraining', 'term': '事前学習', 'aliases': ['pre-training', 'プレトレーニング'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。', 'related': ['fine-tuning', 'llm']}, {'id': 'fine-tuning', 'term': 'ファインチューニング', 'aliases': ['fine-tuning', '追加学習', 'FT'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。', 'related': ['sft', 'peft', 'lora']}, {'id': 'sft', 'term': '教師ありファインチューニング（SFT）', 'aliases': ['SFT', 'Supervised Fine-Tuning', '指示チューニング'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。', 'related': ['fine-tuning', 'lora']}, {'id': 'peft', 'term': 'PEFT', 'aliases': ['Parameter-Efficient Fine-Tuning', 'パラメータ効率の良いファインチューニング'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': 'モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。', 'related': ['lora', 'fine-tuning']}, {'id': 'lora', 'term': 'LoRA', 'aliases': ['Low-Rank Adaptation'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。', 'related': ['peft', 'lora-rank', 'qlora']}, {'id': 'lora-rank', 'term': 'ランク（LoRAのr）', 'aliases': ['rank', 'r'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': 'LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。', 'related': ['lora']}, {'id': 'qlora', 'term': 'QLoRA', 'aliases': ['Quantized LoRA'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': 'ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。', 'related': ['lora', 'quantization']}, {'id': 'quantization', 'term': '量子化', 'aliases': ['quantization'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': 'モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。', 'related': ['qlora']}, {'id': 'learning-rate', 'term': '学習率', 'aliases': ['learning rate', 'lr'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。', 'related': ['epoch', 'overfitting']}, {'id': 'epoch', 'term': 'エポック', 'aliases': ['epoch'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。', 'related': ['overfitting', 'learning-rate']}, {'id': 'overfitting', 'term': '過学習', 'aliases': ['overfitting', 'オーバーフィッティング'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。', 'related': ['epoch', 'evaluation-loop']}, {'id': 'huggingface-hub', 'term': 'Hugging Face Hub', 'aliases': ['Hugging Face', 'HF Hub'], 'lecture': 2, 'category': 'ファインチューニング', 'definition': '公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。', 'related': ['lora', 'pretraining']}, {'id': 'ai-agent', 'term': 'AIエージェント', 'aliases': ['エージェント', 'agent'], 'lecture': 3, 'category': 'エージェント', 'definition': 'LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。', 'related': ['agent-loop', 'tool', 'harness']}, {'id': 'agent-loop', 'term': 'エージェントループ', 'aliases': ['agent loop', 'ツール呼び出しループ'], 'lecture': 3, 'category': 'エージェント', 'definition': 'モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。', 'related': ['ai-agent', 'tool', 'rate-limit']}, {'id': 'tool', 'term': 'ツール（Tool use / Function calling）', 'aliases': ['ツール呼び出し', 'Function calling', 'Tool use', 'ツール'], 'lecture': 3, 'category': 'エージェント', 'definition': 'モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。', 'related': ['tool-description', 'agent-loop']}, {'id': 'tool-description', 'term': 'ツールの説明文', 'aliases': ['docstring', 'ツールスキーマ', 'tool schema'], 'lecture': 3, 'category': 'エージェント', 'definition': 'ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。', 'related': ['tool', 'pydantic-ai']}, {'id': 'structured-output', 'term': '構造化出力', 'aliases': ['structured output', 'JSON出力'], 'lecture': 3, 'category': 'エージェント', 'definition': 'モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。', 'related': ['pydantic-ai']}, {'id': 'harness', 'term': 'ハーネス', 'aliases': ['agent harness', 'エージェントハーネス'], 'lecture': 3, 'category': 'エージェント', 'definition': 'LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。', 'related': ['coding-agent', 'pi']}, {'id': 'pi', 'term': 'Pi', 'aliases': ['Pi coding agent', 'pi agent'], 'lecture': 3, 'category': 'エージェント', 'definition': 'ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。', 'related': ['harness', 'coding-agent']}, {'id': 'pydantic-ai', 'term': 'Pydantic AI', 'aliases': ['pydantic-ai', 'PydanticAI'], 'lecture': 3, 'category': 'エージェント', 'definition': 'Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。', 'related': ['test-model', 'structured-output', 'adk']}, {'id': 'adk', 'term': 'ADK（Agent Development Kit）', 'aliases': ['ADK', 'Agent Development Kit', 'google-adk'], 'lecture': 3, 'category': 'エージェント', 'definition': 'Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。', 'related': ['pydantic-ai', 'ai-agent']}, {'id': 'mcp', 'term': 'MCP（Model Context Protocol）', 'aliases': ['MCP', 'Model Context Protocol'], 'lecture': 3, 'category': 'エージェント', 'definition': 'AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。', 'related': ['tool', 'ai-agent', 'mcp-server', 'mcp-client']}, {'id': 'guardrail', 'term': 'ガードレール', 'aliases': ['guardrail'], 'lecture': 3, 'category': 'エージェント', 'definition': 'エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。', 'related': ['human-in-the-loop', 'system-prompt']}, {'id': 'human-in-the-loop', 'term': 'Human-in-the-loop', 'aliases': ['HITL', '人間による承認', '確認を挟む'], 'lecture': 3, 'category': 'エージェント', 'definition': 'エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。', 'related': ['guardrail', 'tool']}, {'id': 'test-model', 'term': 'TestModel', 'aliases': ['テスト用モデル'], 'lecture': 3, 'category': 'エージェント', 'definition': 'Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。', 'related': ['pydantic-ai', 'rate-limit']}, {'id': 'homework-levels', 'term': '宿題の3レベル', 'aliases': ['3レベル', '宿題レベル'], 'lecture': 1, 'category': '講座', 'definition': 'AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。', 'related': ['answer-md']}, {'id': 'answer-md', 'term': 'answer.md', 'aliases': ['解答ファイル'], 'lecture': 1, 'category': '講座', 'definition': 'AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。', 'related': ['pull-request', 'homework-levels']}, {'id': 'lec-start-tag', 'term': 'lecNN-start タグ', 'aliases': ['開始タグ', 'lec03-start'], 'lecture': 1, 'category': '講座', 'definition': 'AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。', 'related': ['branch']}, {'id': 'default-model', 'term': '既定モデル', 'aliases': ['デフォルトモデル', 'gemini-3.5-flash-lite'], 'lecture': 1, 'category': '講座', 'definition': 'AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。', 'related': ['rate-limit', 'api-key']}, {'id': 'external-tool', 'term': '外部ツール', 'aliases': ['external tool', '外部サービス'], 'lecture': 5, 'category': 'MCP', 'definition': 'Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。', 'related': ['tool', 'mcp', 'mcp-server']}, {'id': 'mcp-server', 'term': 'MCPサーバー', 'aliases': ['MCP server'], 'lecture': 5, 'category': 'MCP', 'definition': 'MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。', 'related': ['mcp', 'mcp-client', 'mcp-transport', 'fetch-server']}, {'id': 'mcp-client', 'term': 'MCPクライアント', 'aliases': ['MCP client', 'MCPホスト'], 'lecture': 5, 'category': 'MCP', 'definition': 'MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。', 'related': ['mcp', 'mcp-server', 'mcp-config']}, {'id': 'mcp-transport', 'term': 'トランスポート（stdio・Streamable HTTP）', 'aliases': ['stdio', 'Streamable HTTP', 'transport', 'MCPのトランスポート'], 'lecture': 5, 'category': 'MCP', 'definition': 'MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。', 'related': ['mcp-server', 'mcp-client']}, {'id': 'mcp-config', 'term': 'mcpServers設定', 'aliases': ['mcpServers', 'MCPの設定ファイル', 'mcp_servers.json', 'mcp.json'], 'lecture': 5, 'category': 'MCP', 'definition': '使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。', 'related': ['mcp-client', 'tool-namespace']}, {'id': 'tool-namespace', 'term': 'ツールの名前空間', 'aliases': ['ツール名のプレフィックス', 'prefix', '名前の衝突'], 'lecture': 5, 'category': 'MCP', 'definition': '複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。', 'related': ['mcp-config', 'tool-description', 'rate-limit']}, {'id': 'fetch-server', 'term': 'fetchサーバー', 'aliases': ['mcp-server-fetch', 'fetch', 'Web取得ツール'], 'lecture': 5, 'category': 'MCP', 'definition': '指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。', 'related': ['mcp-server', 'external-tool', 'guardrail']}]}

TERMS = GLOSSARY["terms"]
BY_ID = {e["id"]: e for e in TERMS}
SOURCE = {
    "repository": "ykuma777/lecture-ai-engineering-2026", "ref": "lec02",
    "glossary_blob_sha": "ab979596e101ed442db4dc8bcfb9b478dd1a3d43",
    "dataset_script_blob_sha": "7e53cfc38f0b70f0141b6f98eab18a94c16a2645",
    "search_tools_blob_sha": "35344270b5859eaecbcd9511add8d11f1bb0fb76",
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("用語数:", len(TERMS))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 3,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "warmup_steps": 6,
  "max_seq_len": 512,
  "max_new_tokens": 160,
  "gen": {
    "do_sample": false
  }
}
用語数: 58


## 2. JSONLをアップロードし、評価用の質問と正解を用意する

Codespacesで作った glossary_sft.jsonl を選択してください。元スクリプトと同じ116件かを確認します。

- 学習：58語×2つの質問テンプレート＝116件。元の質問と定義を使用。
- 評価：5語について直接的な質問5問＋言い換え5問、用語集外5問、合計15問。
- 評価の質問文は学習用と完全一致しないことを確認。既知5語の定義そのものは学習しており、未学習の事実を問うテストではありません。
- 用語集外のうちCS立体図・GeoPackage・カルマンフィルタは実在する用語ですが、この用語集の範囲外です。残り2語は課題用の架空語です。
- この課題では用語集の範囲に従うかを観察します。実在語への回答が用語集外だからといって、回答内容まで誤りだとは断定しません。

正解はキーワード群で用意し、モデルの入力には渡しません。自動採点は意味全体の正しさを保証しないため、最終的には原文も読みます。

In [4]:
from google.colab import files
print("Codespacesで作成した glossary_sft.jsonl をアップロードしてください")
uploaded = files.upload()
assert len(uploaded) == 1, "glossary_sft.jsonlを1つ選んでください"
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
records = [json.loads(line) for line in uploaded_bytes.decode("utf-8-sig").splitlines() if line.strip()]
TEMPLATES = ["「{term}」とは何ですか？", "{term}について簡単に説明してください。"]
expected = [{"messages": [{"role": "user", "content": template.format(term=t["term"])},
                          {"role": "assistant", "content": t["definition"]}]}
            for t in TERMS for template in TEMPLATES]
assert records == expected, "用語集・スクリプトの版またはファイルが異なります。ここで止めて結果を共有してください"
(WORK / "glossary_sft.jsonl").write_bytes(uploaded_bytes)
(WORK / "glossary.json").write_text(json.dumps(GLOSSARY, ensure_ascii=False, indent=2), encoding="utf-8")

GROUPS = {
    "token": [["単位"], ["分割", "断片"]],
    "sft": [["入力", "質問"], ["出力", "回答"], ["ペア", "組", "手本"]],
    "peft": [["一部", "小さ", "少数"], ["学習", "調整"]],
    "lora": [["固定"], ["行列", "低ランク"]],
    "qlora": [["量子化"], ["LoRA"], ["4ビット", "4bit", "４ビット"]],
}
DIRECT = {k: f"用語『{BY_ID[k]['term']}』の意味を教えてください" for k in GROUPS}
REPHRASED = {
    "token": "tokenとは、文章をどのような単位で扱う考え方ですか",
    "sft": "SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか",
    "peft": "PEFTはモデルのどの部分を学習して、計算資源を節約しますか",
    "lora": "LoRAでは元の重みをどう扱い、更新分をどう表しますか",
    "qlora": "QLoRAはLoRAにどのようなメモリ削減の工夫を加えますか",
}
TESTS = []
for category, questions in [("direct", DIRECT), ("rephrased", REPHRASED)]:
    for term_id, question in questions.items():
        TESTS.append({"category": category, "term_id": term_id, "question": question,
                      "in_glossary": True, "gold_definition": BY_ID[term_id]["definition"],
                      "keyword_groups": GROUPS[term_id]})
for term, fictional in [("CS立体図", False), ("GeoPackage", False), ("カルマンフィルタ", False),
                        ("ブルーリーフ最適化", True), ("星砂アダプタ", True)]:
    TESTS.append({"category": "out_of_glossary", "term_id": term,
                  "question": f"用語『{term}』の意味を教えてください", "in_glossary": False,
                  "fictional_term": fictional, "gold_definition": "この用語集には記載がありません。確認が必要です。", "keyword_groups": []})
train_questions = {r["messages"][0]["content"] for r in records}
assert len(TESTS) == 15 and len({t["question"] for t in TESTS}) == 15
assert not train_questions.intersection(t["question"] for t in TESTS), "評価質問が学習質問と重複しています"
(WORK / "test_and_gold.json").write_text(json.dumps(TESTS, ensure_ascii=False, indent=2), encoding="utf-8")

# app/tools.py の3関数をそのまま取り出した検索。質問全文を検索語に使い、正解の用語IDは渡しません。
MAX_RESULTS = 3
def load_glossary(): return TERMS

def _normalize(text: str) -> str:
    return text.casefold().replace(' ', '').replace('\u3000', '')

def _names(entry: dict) -> list[str]:
    return [_normalize(entry['term']), _normalize(entry['id'])] + [_normalize(a) for a in entry['aliases']]

def search_glossary(query: str) -> str:
    """講座の用語集から用語を検索し、定義と、どの回で扱うかを返す。

    AIエンジニアリングの用語や、この講座（AIE2026）に固有の言葉について
    答える前に使うこと。query には調べたい用語を1つだけ入れる（例: "LoRA"）。
    """
    q = _normalize(query)
    if not q:
        return '検索語が空です。調べたい用語を1つ指定してください。'
    glossary = load_glossary()
    exact = [e for e in glossary if q in _names(e)]
    partial = [e for e in glossary if e not in exact and any((q in n or (len(n) >= 3 and n in q) for n in _names(e)))]
    in_definition = [e for e in glossary if e not in exact and e not in partial and (q in _normalize(e['definition']))]
    hits = (exact + partial + in_definition)[:MAX_RESULTS]
    if not hits:
        return f'用語集に「{query}」は見つかりませんでした。'
    return json.dumps([{'term': e['term'], 'lecture': e['lecture'], 'definition': e['definition'], 'related': e['related']} for e in hits], ensure_ascii=False)

print("学習件数:", len(records), "| 評価件数:", len(TESTS))
print("アップロード元:", uploaded_name, "| SHA-256:", hashlib.sha256(uploaded_bytes).hexdigest())
for t in TESTS:
    text = search_glossary(t["question"])
    try: hits = json.loads(text)
    except ValueError: hits = []
    print(t["category"], t["question"], "→ 検索:", [h["term"] for h in hits])

Codespacesで作成した glossary_sft.jsonl をアップロードしてください


Saving glossary_sft.jsonl to glossary_sft.jsonl
学習件数: 116 | 評価件数: 15
アップロード元: glossary_sft.jsonl | SHA-256: af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081
direct 用語『トークン』の意味を教えてください → 検索: ['トークン']
direct 用語『教師ありファインチューニング（SFT）』の意味を教えてください → 検索: ['ファインチューニング', '教師ありファインチューニング（SFT）']
direct 用語『PEFT』の意味を教えてください → 検索: ['PEFT']
direct 用語『LoRA』の意味を教えてください → 検索: ['LoRA']
direct 用語『QLoRA』の意味を教えてください → 検索: ['LoRA', 'QLoRA']
rephrased tokenとは、文章をどのような単位で扱う考え方ですか → 検索: ['トークン']
rephrased SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか → 検索: ['教師ありファインチューニング（SFT）']
rephrased PEFTはモデルのどの部分を学習して、計算資源を節約しますか → 検索: ['PEFT']
rephrased LoRAでは元の重みをどう扱い、更新分をどう表しますか → 検索: ['LoRA']
rephrased QLoRAはLoRAにどのようなメモリ削減の工夫を加えますか → 検索: ['LoRA', 'QLoRA']
out_of_glossary 用語『CS立体図』の意味を教えてください → 検索: []
out_of_glossary 用語『GeoPackage』の意味を教えてください → 検索: []
out_of_glossary 用語『カルマンフィルタ』の意味を教えてください → 検索: []
out_of_glossary 用語『ブルーリーフ最適化』の意味を教えてください → 検索: []
out_of_glossary 用語『星砂アダプタ』の意味を教えてください → 検索: []


## 3. 学習前：用語集を入力に渡さず15問を評価する

プロンプトは共通のsystem指示と質問だけです。既知語は重要語句の検出率、用語集外は確認要求の表現が出た割合を測ります。
空の回答や自信のない回答も原文で確認してください。ここではまだ用語集を検索しません。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


def make_messages(question, reference=None):
    user = question
    if reference is not None:
        user = f"用語集の検索結果（参照資料）:\n{reference}\n\n質問: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate_answer(active_model, question, reference=None):
    prompt = tok.apply_chat_template(make_messages(question, reference), tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(active_model.device)
    with torch.no_grad():
        out = active_model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"],
                                    pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
_ = model.eval()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

def normalize_answer(s):
    return unicodedata.normalize("NFKC", s).casefold().replace(" ", "").replace("\n", "")

def score_answer(raw, item):
    text = normalize_answer(raw)
    review = any(normalize_answer(w) in text for w in ["確認", "記載がありません", "記載されていません", "見つかりません", "分かりません", "わかりません", "知りません", "未確認"])
    groups_ok = all(any(normalize_answer(w) in text for w in group) for group in item["keyword_groups"])
    # 既知語：重要語句が揃い、確認要求に逃げていないこと。未知語：確認要求語があること。
    proxy_ok = (groups_ok and not review and bool(raw)) if item["in_glossary"] else review
    return {"review_signal": review, "definition_keywords_ok": groups_ok if item["in_glossary"] else None,
            "task_proxy_ok": proxy_ok}

def evaluate(active_model, label, use_search=False):
    rows = []
    started = time.time()
    for item in TESTS:
        reference = search_glossary(item["question"]) if use_search else None
        raw = generate_answer(active_model, item["question"], reference)
        rows.append({"question": item["question"], "category": item["category"],
                     "in_glossary": item["in_glossary"], "raw": raw,
                     "retrieved_reference": reference, **score_answer(raw, item)})
    summary = {"label": label, "n": len(rows), "seconds": round(time.time()-started, 1)}
    for cat in ["direct", "rephrased", "out_of_glossary"]:
        part = [r for r in rows if r["category"] == cat]
        summary[cat + "_proxy_rate"] = sum(r["task_proxy_ok"] for r in part) / len(part)
    summary["all_proxy_rate"] = sum(r["task_proxy_ok"] for r in rows) / len(rows)
    summary["known_review_rate"] = sum(r["review_signal"] for r in rows if r["in_glossary"]) / 10
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    display(pd.DataFrame(rows)[["category", "question", "raw", "review_signal", "task_proxy_ok"]])
    return summary, rows

before_summary, before_rows = evaluate(model, "before")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{
  "label": "before",
  "n": 15,
  "seconds": 102.6,
  "direct_proxy_rate": 0.0,
  "rephrased_proxy_rate": 0.2,
  "out_of_glossary_proxy_rate": 0.0,
  "all_proxy_rate": 0.06666666666666667,
  "known_review_rate": 0.0
}


,category,question,raw,review_signal,task_proxy_ok
0,direct,用語『トークン』の意味を教えてください,トークンは、言語の構造や表現において「単位」として扱われる基本の言葉です。\n\n語句を構成する基本単位として、**「単語（Word）」**と**「文法（Grammar）」**を区別します。\n* **単語**は、語句の基本単位であり、語句の構成要素として扱われます。\n* **文法**は、単語をどのように組み合わせるかを規定する規則やルールです。\n\nしたがって、トークン（単語）は、文法によって定義された基本の言葉単位であり、その基本単位として扱われます。,False,False
1,direct,用語『教師ありファインチューニング（SFT）』の意味を教えてください,教師ありファインチューニング（SFT）は、**「教師（データ）から学習（モデル）を学習させる」**という概念を指します。\n\n具体的には、教師のデータ（例：「猫は白い」「猫は黒い」など）を学習する際に、学習モデル（ニューラルネットワークなど）が「教師のデータ」を参照して、**「教師のデータから学習した知識」**を学習するプロセスです。\n\nこのプロセスは、**教師なし学習（SFT）とは対照的に、教師のデータが学習対象（入力）として直接入力されるため、学習がより確実で安定する**という特徴を持っています。,False,False
2,direct,用語『PEFT』の意味を教えてください,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの学習を**パラメータ効率**で加速する手法です。\n\nこの用語は、以下の 3 つの要素を組み合わせて使われます：\n1. **Parameter**：モデルの**パラメータ**（学習対象の知識や特徴）\n2. **Efficient**：効率的（学習速度を上げるため）\n3. **Fine-Tuning**：**タスク**（学習目的）に合わせて調整する\n\nつまり、モデルの知識を学習する際に、**「パラメータを減らす（効率化）」**ことで、学習速度を上げ、より多くのデータや計算時間を節約できることを指します。\n\n※用語集に記載されていない場合は、推測,False,False
3,direct,用語『LoRA』の意味を教えてください,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の**パラメータを「低秩」に削減し、より軽量かつ高速に学習させる技術**です。\n\n主な特徴は以下の通りです：\n\n* **低秩性（Low-Rankness）**: モデルのパラメータを「低秩」に分解し、その分解された部分だけを学習対象にします。\n* **学習効率**: 学習するパラメータの数を大幅に減らし、計算コストを削減します。\n* **可視化**: 学習したパラメータの構造や変化を、モデルの出力結果（例：生成された文章の文脈や単語の分布）から**可視化,False,False
4,direct,用語『QLoRA』の意味を教えてください,QLoRA は、**「Quantum Large Language Model（量子大規模言語モデル）」**の略です。\n\nこの用語は、主に**Google**が開発した、大規模言語モデル（LLM）の一種です。その特徴は以下の通りです。\n\n* **量子計算の基盤**: 従来の機械学習モデルでは計算コストが非常に高く、特に大規模モデルでは計算量が膨大になります。QLoRA は、量子計算の原理（量子ビット、量子重ね合わせなど）を基盤に設計されたモデルです。\n* **計算効率**: 従来のモデルでは計算量が膨大になりすぎて、処理速度が非常に遅く、実用上では使用できません。QLoRA は、量子計算の効率化により、**,False,False
5,rephrased,tokenとは、文章をどのような単位で扱う考え方ですか,文章を**文字列（テキスト）**単位で扱う考え方です。\n\n具体的には、テキストファイルやテキストエディタなどの文脈で、文字列を扱うことを指します。,False,False
6,rephrased,SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか,SFT（ supervised fine-tuning）でモデルに学習させる際、入力と正解の組は以下の通りです。\n\n* **入力 (Input):** 学習対象のテキストやコード、画像などのデータ。\n* **正解 (Label):** 入力に対して正しい答え（または正解のテキスト、画像など）を指定する。\n\nこの組み合わせを「学習データ」として、モデルを訓練します。,False,False
7,rephrased,PEFTはモデルのどの部分を学習して、計算資源を節約しますか,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**パラメータ（weights）を学習**し、計算資源を節約します。\n\n特に、モデルの**多数のパラメータを学習するのではなく、少数のパラメータ（特にロジックや特徴的な部分）を学習**することで、学習コストを大幅に削減できます。,False,True
8,rephrased,LoRAでは元の重みをどう扱い、更新分をどう表しますか,LoRA（低精度リファインリング）では、元の重みを**そのまま維持**し、更新分を**ゼロ**にします。\n\nつまり、LoRA 学習は「元のモデルをそのまま維持しつつ、必要な部分だけを修正する」アプローチであり、元の重みはそのまま残ります。,False,False
9,rephrased,QLoRAはLoRAにどのようなメモリ削減の工夫を加えますか,QLoRA は、LoRA のメモリ削減に以下の工夫を加えています。\n\n* **パラメータの削減**: 学習率を 100 倍に設定し、計算量を抑えるため、LoRA のパラメータ数を大幅に減らします。\n* **メモリ効率の向上**: 計算量を抑えつつ、メモリ消費も削減することで、より多くのモデルを学習できるため、メモリ効率を向上させます。,False,False


## 4. 用語集の116件でLoRA学習する

元のスクリプトが作った user / assistant の組を使用し、共通のsystem指示だけを加えます。学習時にFAQや用語集全文を入力には渡しません。
3エポック・バッチ4・勾配累積2で約45ステップ。評価の正解や質問文は学習に入れません。lossは低下するかを見ますが、lossだけで実用性を判断しません。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

training_records = [{"messages": [{"role": "system", "content": SYSTEM}] + r["messages"]} for r in records]
train_ds = Dataset.from_list(training_records)
(WORK / "training_messages.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in training_records)+"\n", encoding="utf-8")
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05,
                  bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=5, save_steps=15, save_total_limit=2,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
started = time.time()
trainer.train()
TRAINING_SECONDS = round(time.time()-started, 1)
print("学習時間:", TRAINING_SECONDS, "秒 | ステップ:", trainer.state.global_step)
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
5,3.209525
10,1.907821
15,1.536198
20,1.237267
25,1.126223
30,1.105519
35,0.807766
40,0.874314
45,0.800382


学習時間: 163.0 秒 | ステップ: 45


## 5. 学習後：資料なしで同じ15問を評価する

変えたのはadapterの有無です。質問・system指示・生成設定は同じにします。定義の語句が揃ったか、言い換えで崩れたか、用語集外に断定したかを読みます。

In [8]:
after_summary, after_rows = evaluate(tuned, "after")

{
  "label": "after",
  "n": 15,
  "seconds": 49.5,
  "direct_proxy_rate": 0.6,
  "rephrased_proxy_rate": 0.0,
  "out_of_glossary_proxy_rate": 0.0,
  "all_proxy_rate": 0.2,
  "known_review_rate": 0.0
}


,category,question,raw,review_signal,task_proxy_ok
0,direct,用語『トークン』の意味を教えてください,LLMがテキストを扱う最小単位。単語や文字ではなく、単語の英数値（例：「apple」は5トークン）で割る。トークン数を増やしすぎるとモデルの処理能力を消費し、削減するには分割を大きくする。,False,True
1,direct,用語『教師ありファインチューニング（SFT）』の意味を教えてください,入力と出力のペアでモデルを学習させる方法。質問の形式や返しの内容に強い。本講座の第2回演習では主に使う。,False,True
2,direct,用語『PEFT』の意味を教えてください,モデルの全パラメータを小さなセットのデータで学習させること。全パラメータを学習させるのは計算が重いため、一部分だけを学習させることで学習コストと計算量を大幅に減らせる。,False,True
3,direct,用語『LoRA』の意味を教えてください,低ランクアサインの学習法。重みを増やし、学習率を小さくする。学習するパラメータは、重みと低ランクアサインの和だけ。,False,False
4,direct,用語『QLoRA』の意味を教えてください,LLMがメモリ使用量に制限がある場合、学習した重みをより小さなメモリを持つモデルにコピーする技術。本講座では「LoRA」という手法で、重みを1つのベクトルにまとめることでメモリを節約できる。,False,False
5,rephrased,tokenとは、文章をどのような単位で扱う考え方ですか,LLMが受け取る入力と、出力する結果を切り分ける単位。トークン数が多いほど、入力と出力の境界が明確になり、モデルの挙動やコスト、検索の精度に影響する。,False,False
6,rephrased,SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか,教師の例（instruction + answer）のペア。例：「単語を日本語で説明してください」と答えて「単語の意味を日本語で説明してください」と返す。モデルはこのペアから、次の入力（次の質問）と、その質問に対する答えを予測する。,False,False
7,rephrased,PEFTはモデルのどの部分を学習して、計算資源を節約しますか,モデルの全パラメータを学習せず、特定の部分（低ランク）だけを学習する手法。全パラメータを学習する場合はGPUメモリや計算量に大きく依存し、低ランクを学習する場合はメモリや計算量に依存し、より少ない資源でモデルを学習できる。,False,False
8,rephrased,LoRAでは元の重みをどう扱い、更新分をどう表しますか,LoRAでは、学習した重みを0に直す（重みを消す）のではなく、元の重みをそのまま使う。更新分は、元の重みと新しい重みの差（低ランクのベクトル）を2つの低ランクベクトルで表す。,False,False
9,rephrased,QLoRAはLoRAにどのようなメモリ削減の工夫を加えますか,LoRAの重みを1つのメモリで持てないため、重みを小さな行列（例：16ビットの256×256）で表し、その行列を2つの小さな行列（LoRA weightとLoRA bias）に分けてメモリを節約する。本講座ではLoRA weightを16ビットで256×256の行列とみなし、2つの小さな行列をメモリで扱う。,False,False


## 6. 検索あり：adapterを外し、用語集の検索結果を渡す

この条件は追加学習前のモデルで行います。質問全文を講座のsearch_glossaryへ渡し、見つかった上位3件までの定義を入力します。正解の用語IDを検索に渡しません。

検索あり条件では入力が増えるため、資料なし条件と入力長・待ち時間が違います。検索の失敗と、資料を読んだ後の生成の失敗を区別してください。検索結果に答えがあっても正しく回答できるとは限りません。

In [9]:
with tuned.disable_adapter():
    rag_summary, rag_rows = evaluate(tuned, "base_with_search", use_search=True)
display(pd.DataFrame([{"質問": r["question"], "検索結果": r["retrieved_reference"], "回答": r["raw"]} for r in rag_rows]))

{
  "label": "base_with_search",
  "n": 15,
  "seconds": 45.1,
  "direct_proxy_rate": 0.8,
  "rephrased_proxy_rate": 1.0,
  "out_of_glossary_proxy_rate": 0.8,
  "all_proxy_rate": 0.8666666666666667,
  "known_review_rate": 0.0
}


,category,question,raw,review_signal,task_proxy_ok
0,direct,用語『トークン』の意味を教えてください,LLM（大規模言語モデル）がテキストを扱う最小単位です。\nAPI の料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数えます。\n日本語は英語より 1 文字あたりのトークン数が多くなりやすいです。,False,False
1,direct,用語『教師ありファインチューニング（SFT）』の意味を教えてください,教師ありファインチューニング（SFT）は、**「質問と模範回答のペア（入力・出力）を使ってモデルを追加学習する」**という方法です。\n\nこの手法は、モデルが指示に従う振る舞いを身につけさせる代表的な手法であり、学習の方向性が明確な「指示」に基づいています。,False,True
2,direct,用語『PEFT』の意味を教えてください,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**全パラメータを学習するのではなく、ごく一部や追加した小さなパラメータだけを学習する手法**の総称です。\n\nこの手法は、必要な**GPU メモリと保存容量を大きく減らす**ため、学習速度を向上させます。LoRA（Low-Rank Adaptation）が代表的な例です。,False,True
3,direct,用語『LoRA』の意味を教えてください,LoRA（Low-Rank Adaptation）は、学習済みの重み（パラメータ）を固定し、学習分を「低ランク行列」で補完して学習する手法です。\n\nこの手法は、学習するパラメータが少なく、学習結果（アダプタ）も小さく保存できるため、効率的に学習できます。,False,True
4,direct,用語『QLoRA』の意味を教えてください,QLoRA は、ベースモデルの重みを 4 ビットなどの量子化（低ランク）してメモリを節約し、LoRA によって学習する手法です。,False,True
5,rephrased,tokenとは、文章をどのような単位で扱う考え方ですか,トークンは、LLM（大規模言語モデル）がテキストを扱う最小単位です。\nこれは、単語や文字の断片に分割されたものです。\nAPI の料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数えます。\n日本語は英語より 1 文字あたりのトークン数が多くなりやすいです。,False,True
6,rephrased,SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか,SFT（教師ありファインチューニング）でモデルに学ばせる際、入力と正解の組は以下の通りです。\n\n* **入力**: 質問（または問題文）\n* **正解**: 模範回答（または正解の答え）,False,True
7,rephrased,PEFTはモデルのどの部分を学習して、計算資源を節約しますか,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**全パラメータ**ではなく、ごく一部や追加した小さなパラメータだけを学習する手法です。\n\n計算資源を節約する主な理由は、GPU メモリと保存容量を大きく減らすためです。,False,True
8,rephrased,LoRAでは元の重みをどう扱い、更新分をどう表しますか,LoRA では、元の重みを**固定**にします。\n学習するパラメータは、元の重みと**更新分**の積（低ランク行列）で表し、その小さな行列だけを学習します。,False,True
9,rephrased,QLoRAはLoRAにどのようなメモリ削減の工夫を加えますか,QLoRA は LoRA の重みを **4 ビット** に量子化してメモリを節約し、より大きなモデルを小さな GPU で扱える工夫です。,False,True


,質問,検索結果,回答
0,用語『トークン』の意味を教えてください,"[{""term"": ""トークン"", ""lecture"": 1, ""definition"": ""LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。"", ""related"": [""context-window"", ""rate-limit""]}]",LLM（大規模言語モデル）がテキストを扱う最小単位です。\nAPI の料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数えます。\n日本語は英語より 1 文字あたりのトークン数が多くなりやすいです。
1,用語『教師ありファインチューニング（SFT）』の意味を教えてください,"[{""term"": ""ファインチューニング"", ""lecture"": 2, ""definition"": ""事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。"", ""related"": [""sft"", ""peft"", ""lora""]}, {""term"": ""教師ありファインチューニング（SFT）"", ""lecture"": 2, ""definition"": ""入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。"", ""related"": [""fine-tuning"", ""lora""]}]",教師ありファインチューニング（SFT）は、**「質問と模範回答のペア（入力・出力）を使ってモデルを追加学習する」**という方法です。\n\nこの手法は、モデルが指示に従う振る舞いを身につけさせる代表的な手法であり、学習の方向性が明確な「指示」に基づいています。
2,用語『PEFT』の意味を教えてください,"[{""term"": ""PEFT"", ""lecture"": 2, ""definition"": ""モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。"", ""related"": [""lora"", ""fine-tuning""]}]",PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**全パラメータを学習するのではなく、ごく一部や追加した小さなパラメータだけを学習する手法**の総称です。\n\nこの手法は、必要な**GPU メモリと保存容量を大きく減らす**ため、学習速度を向上させます。LoRA（Low-Rank Adaptation）が代表的な例です。
3,用語『LoRA』の意味を教えてください,"[{""term"": ""LoRA"", ""lecture"": 2, ""definition"": ""事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。"", ""related"": [""peft"", ""lora-rank"", ""qlora""]}]",LoRA（Low-Rank Adaptation）は、学習済みの重み（パラメータ）を固定し、学習分を「低ランク行列」で補完して学習する手法です。\n\nこの手法は、学習するパラメータが少なく、学習結果（アダプタ）も小さく保存できるため、効率的に学習できます。
4,用語『QLoRA』の意味を教えてください,"[{""term"": ""LoRA"", ""lecture"": 2, ""definition"": ""事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。"", ""related"": [""peft"", ""lora-rank"", ""qlora""]}, {""term"": ""QLoRA"", ""lecture"": 2, ""definition"": ""ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。"", ""related"": [""lora"", ""quantization""]}]",QLoRA は、ベースモデルの重みを 4 ビットなどの量子化（低ランク）してメモリを節約し、LoRA によって学習する手法です。
5,tokenとは、文章をどのような単位で扱う考え方ですか,"[{""term"": ""トークン"", ""lecture"": 1, ""definition"": ""LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。"", ""related"": [""context-window"", ""rate-limit""]}]",トークンは、LLM（大規模言語モデル）がテキストを扱う最小単位です。\nこれは、単語や文字の断片に分割されたものです。\nAPI の料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数えます。\n日本語は英語より 1 文字あたりのトークン数が多くなりやすいです。
6,SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか,"[{""term"": ""教師ありファインチューニング（SFT）"", ""lecture"": 2, ""definition"": ""入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。"", ""related"": [""fine-tuning"", ""lora""]}]",SFT（教師ありファインチューニング）でモデルに学ばせる際、入力と正解の組は以下の通りです。\n\n* **入力**: 質問（または問題文）\n* **正解**: 模範回答（または正解の答え）
7,PEFTはモデルのどの部分を学習して、計算資源を節約しますか,"[{""term"": ""PEFT"", ""lecture"": 2, ""definition"": ""モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。"", ""related"": [""lora"", ""fine-tuning""]}]",PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**全パラメータ**ではなく、ごく一部や追加した小さなパラメータだけを学習する手法です。\n\n計算資源を節約する主な理由は、GPU メモリと保存容量を大きく減らすためです。
8,LoRAでは元の重みをどう扱い、更新分をどう表しますか,"[{""term"": ""LoRA"", ""lecture"": 2, ""definition"": ""事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。"", ""related"": [""peft"", ""lora-rank"", ""qlora""]}]",LoRA では、元の重みを**固定**にします。\n学習するパラメータは、元の重みと**更新分**の積（低ランク行列）で表し、その小さな行列だけを学習します。
9,QLoRAはLoRAにどのようなメモリ削減の工夫を加えますか,"[{""term"": ""LoRA"", ""lecture"": 2, ""definition"": ""事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。"", ""related"": [""peft"", ""lora-rank"", ""qlora""]}, {""term"": ""QLoRA"", ""lecture"": 2, ""defi

## 7. 比較と保存

proxy_rateはキーワード・確認要求語による自動判定の目安です。点数だけで良し悪しを結論にせず、同じ質問の原文を見て判断します。誤った説明に重要語句が含まれる場合も、正しい言い換えが語句検出に落ちる場合もあります。

見る観点：①既知語の定義 ②言い換えへの対応 ③用語集外の確認要求 ④検索結果の有無 ⑤学習の手間と資料を毎回渡す手間。

FTで扱う知識と、検索で渡す情報の違いを考察します。今回の実在の用語集外語への断定は、用語集という範囲への違反であって、必ずしも事実として誤りとは限りません。

In [10]:
comparison = pd.DataFrame([before_summary, after_summary, rag_summary]).set_index("label")
display(comparison)
detail = [{"質問": b["question"], "種類": b["category"],
           "学習前": b["raw"], "学習後": a["raw"], "検索あり": r["raw"]}
          for b, a, r in zip(before_rows, after_rows, rag_rows)]
display(pd.DataFrame(detail))
loss_log = [r for r in trainer.state.log_history if "loss" in r]
result = {"config": CONFIG, "source": SOURCE,
          "input_jsonl_sha256": hashlib.sha256(uploaded_bytes).hexdigest(),
          "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
          "dataset": {"terms": len(TERMS), "training_records": len(records), "evaluation_questions": len(TESTS)},
          "training": {"seconds": TRAINING_SECONDS, "steps": trainer.state.global_step,
                       "final_logged_loss": loss_log[-1]["loss"] if loss_log else None,
                       "log_history": trainer.state.log_history},
          "before": before_summary, "after": after_summary, "base_with_search": rag_summary,
          "metric_note": "重要語句と確認要求語の検出による代理指標。意味的な正しさは原文確認が必要。",
          "executed_training_myself": True}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "comparison_detail.json").write_text(json.dumps(detail, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))

,n,seconds,direct_proxy_rate,rephrased_proxy_rate,out_of_glossary_proxy_rate,all_proxy_rate,known_review_rate
label,,,,,,,
before,15,102.6,0.0,0.2,0.0,0.066667,0.0
after,15,49.5,0.6,0.0,0.0,0.200000,0.0
base_with_search,15,45.1,0.8,1.0,0.8,0.866667,0.0


,質問,種類,学習前,学習後,検索あり
0,用語『トークン』の意味を教えてください,direct,トークンは、言語の構造や表現において「単位」として扱われる基本の言葉です。\n\n語句を構成する基本単位として、**「単語（Word）」**と**「文法（Grammar）」**を区別します。\n* **単語**は、語句の基本単位であり、語句の構成要素として扱われます。\n* **文法**は、単語をどのように組み合わせるかを規定する規則やルールです。\n\nしたがって、トークン（単語）は、文法によって定義された基本の言葉単位であり、その基本単位として扱われます。,LLMがテキストを扱う最小単位。単語や文字ではなく、単語の英数値（例：「apple」は5トークン）で割る。トークン数を増やしすぎるとモデルの処理能力を消費し、削減するには分割を大きくする。,LLM（大規模言語モデル）がテキストを扱う最小単位です。\nAPI の料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数えます。\n日本語は英語より 1 文字あたりのトークン数が多くなりやすいです。
1,用語『教師ありファインチューニング（SFT）』の意味を教えてください,direct,教師ありファインチューニング（SFT）は、**「教師（データ）から学習（モデル）を学習させる」**という概念を指します。\n\n具体的には、教師のデータ（例：「猫は白い」「猫は黒い」など）を学習する際に、学習モデル（ニューラルネットワークなど）が「教師のデータ」を参照して、**「教師のデータから学習した知識」**を学習するプロセスです。\n\nこのプロセスは、**教師なし学習（SFT）とは対照的に、教師のデータが学習対象（入力）として直接入力されるため、学習がより確実で安定する**という特徴を持っています。,入力と出力のペアでモデルを学習させる方法。質問の形式や返しの内容に強い。本講座の第2回演習では主に使う。,教師ありファインチューニング（SFT）は、**「質問と模範回答のペア（入力・出力）を使ってモデルを追加学習する」**という方法です。\n\nこの手法は、モデルが指示に従う振る舞いを身につけさせる代表的な手法であり、学習の方向性が明確な「指示」に基づいています。
2,用語『PEFT』の意味を教えてください,direct,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの学習を**パラメータ効率**で加速する手法です。\n\nこの用語は、以下の 3 つの要素を組み合わせて使われます：\n1. **Parameter**：モデルの**パラメータ**（学習対象の知識や特徴）\n2. **Efficient**：効率的（学習速度を上げるため）\n3. **Fine-Tuning**：**タスク**（学習目的）に合わせて調整する\n\nつまり、モデルの知識を学習する際に、**「パラメータを減らす（効率化）」**ことで、学習速度を上げ、より多くのデータや計算時間を節約できることを指します。\n\n※用語集に記載されていない場合は、推測,モデルの全パラメータを小さなセットのデータで学習させること。全パラメータを学習させるのは計算が重いため、一部分だけを学習させることで学習コストと計算量を大幅に減らせる。,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**全パラメータを学習するのではなく、ごく一部や追加した小さなパラメータだけを学習する手法**の総称です。\n\nこの手法は、必要な**GPU メモリと保存容量を大きく減らす**ため、学習速度を向上させます。LoRA（Low-Rank Adaptation）が代表的な例です。
3,用語『LoRA』の意味を教えてください,direct,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の**パラメータを「低秩」に削減し、より軽量かつ高速に学習させる技術**です。\n\n主な特徴は以下の通りです：\n\n* **低秩性（Low-Rankness）**: モデルのパラメータを「低秩」に分解し、その分解された部分だけを学習対象にします。\n* **学習効率**: 学習するパラメータの数を大幅に減らし、計算コストを削減します。\n* **可視化**: 学習したパラメータの構造や変化を、モデルの出力結果（例：生成された文章の文脈や単語の分布）から**可視化,低ランクアサインの学習法。重みを増やし、学習率を小さくする。学習するパラメータは、重みと低ランクアサインの和だけ。,LoRA（Low-Rank Adaptation）は、学習済みの重み（パラメータ）を固定し、学習分を「低ランク行列」で補完して学習する手法です。\n\nこの手法は、学習するパラメータが少なく、学習結果（アダプタ）も小さく保存できるため、効率的に学習できます。
4,用語『QLoRA』の意味を教えてください,direct,QLoRA は、**「Quantum Large Language Model（量子大規模言語モデル）」**の略です。\n\nこの用語は、主に**Google**が開発した、大規模言語モデル（LLM）の一種です。その特徴は以下の通りです。\n\n* **量子計算の基盤**: 従来の機械学習モデルでは計算コストが非常に高く、特に大規模モデルでは計算量が膨大になります。QLoRA は、量子計算の原理（量子ビット、量子重ね合わせなど）を基盤に設計されたモデルです。\n* **計算効率**: 従来のモデルでは計算量が膨大になりすぎて、処理速度が非常に遅く、実用上では使用できません。QLoRA は、量子計算の効率化により、**,LLMがメモリ使用量に制限がある場合、学習した重みをより小さなメモリを持つモデルにコピーする技術。本講座では「LoRA」という手法で、重みを1つのベクトルにまとめることでメモリを節約できる。,QLoRA は、ベースモデルの重みを 4 ビットなどの量子化（低ランク）してメモリを節約し、LoRA によって学習する手法です。
5,tokenとは、文章をどのような単位で扱う考え方ですか,rephrased,文章を**文字列（テキスト）**単位で扱う考え方です。\n\n具体的には、テキストファイルやテキストエディタなどの文脈で、文字列を扱うことを指します。,LLMが受け取る入力と、出力する結果を切り分ける単位。トークン数が多いほど、入力と出力の境界が明確になり、モデルの挙動やコスト、検索の精度に影響する。,トークンは、LLM（大規模言語モデル）がテキストを扱う最小単位です。\nこれは、単語や文字の断片に分割されたものです。\nAPI の料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数えます。\n日本語は英語より 1 文字あたりのトークン数が多くなりやすいです。
6,SFTでモデルに学ばせる際、どんな入力と正解の組を使いますか,rephrased,SFT（ supervised fine-tuning）でモデルに学習させる際、入力と正解の組は以下の通りです。\n\n* **入力 (Input):** 学習対象のテキストやコード、画像などのデータ。\n* **正解 (Label):** 入力に対して正しい答え（または正解のテキスト、画像など）を指定する。\n\nこの組み合わせを「学習データ」として、モデルを訓練します。,教師の例（instruction + answer）のペア。例：「単語を日本語で説明してください」と答えて「単語の意味を日本語で説明してください」と返す。モデルはこのペアから、次の入力（次の質問）と、その質問に対する答えを予測する。,SFT（教師ありファインチューニング）でモデルに学ばせる際、入力と正解の組は以下の通りです。\n\n* **入力**: 質問（または問題文）\n* **正解**: 模範回答（または正解の答え）
7,PEFTはモデルのどの部分を学習して、計算資源を節約しますか,rephrased,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**パラメータ（weights）を学習**し、計算資源を節約します。\n\n特に、モデルの**多数のパラメータを学習するのではなく、少数のパラメータ（特にロジックや特徴的な部分）を学習**することで、学習コストを大幅に削減できます。,モデルの全パラメータを学習せず、特定の部分（低ランク）だけを学習する手法。全パラメータを学習する場合はGPUメモリや計算量に大きく依存し、低ランクを学習する場合はメモリや計算量に依存し、より少ない資源でモデルを学習できる。,PEFT（Parameter-Efficient Fine-Tuning）は、モデルの**全パラメータ**ではなく、ごく一部や追加した小さなパラメータだけを学習する手法です。\n\n計算資源を節約する主な理由は、GPU メモリと保存容量を大きく減らすためです。
8,LoRAでは元の重みをどう扱い、更新分をどう表しますか,rephrased,LoRA（低精度リファインリング）では、元の重みを**そのまま維持**し、更新分を**ゼロ**にします。\n\nつまり、LoRA 学習は「元のモデルをそのまま維持しつつ、必要な部分だけを修正する」アプローチであり、元の重みはそのまま残ります。,LoRAでは、学習した重みを0に

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 3,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "warmup_steps": 6,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {
      "do_sample": false
    }
  },
  "source": {
    "repository": "ykuma777/lecture-ai-engineering-2026",
    "ref": "lec02",
    "glossary_blob_sha": "ab979596e101ed442db4dc8bcfb9b478dd1a3d43",
    "dataset_script_blob_sha": "7e53cfc38f0b70f0141b6f98eab18a94c16a2645",
    "search_tools_blob_sha": "35344270b5859eaecbcd9511add8d11f1bb0fb76"
  },
  "input_jsonl_sha256": "af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081",
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "dataset": {
    "terms": 58,
    "training_records": 116,
    "evaluation_questions": 15
  },
  "training": {
    "seconds": 163.0,

## 8. 結果ZIPをダウンロード

結果ZIPを保存し、このノートブックも「ファイル → ダウンロード → .ipynbをダウンロード」で出力を残して保存してください。結果が保存できたらGPUランタイムを接続解除・削除します。

In [11]:
import zipfile
from google.colab import files
download_path = Path("/content/glossary_advanced_results.zip")
names = ["glossary_sft.jsonl", "glossary.json", "training_messages.jsonl", "test_and_gold.json",
         "eval_before.json", "eval_after.json", "eval_base_with_search.json", "result.json", "comparison_detail.json"]
assert all((WORK / name).is_file() for name in names), "上から順に実行してください"
with zipfile.ZipFile(download_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in names: z.write(WORK / name, arcname=name)
files.download(str(download_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>